### QuSciTech-Labs — Navigation

[Public Labs](https://github.com/jopaneur/quscitech-labs) ·
[Full Edition Access](https://github.com/jopaneur/quscitech-labs#-full-edition-kdp) ·
[Private Repo](https://github.com/jopaneur/quscitech-labs-full) ·
[QuSciTech.com](https://www.quscitech.com) ·
[The Quantum AI Book (QAIS)](https://www.amazon.com/dp/placeholder) ·

DOI: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17212825.svg)](https://doi.org/10.5281/zenodo.17212825)

### E.2 Lab 6 — Simple VQE-Style Minimization (Toy) — Converging Energy Curve

### Lab Access and Execution Guide
This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional Volume).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_2_Operations_&_Scientific_Framework_of_QAIS_Advanced_Challenge_Bloch_Trajectories_Under_Composite_Gates.ipynb)


---
**Note for Lab Participants**
Each plot generated in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Book Reference: Chapter 8 — **

Ch08 — Optimization and Control in Quantum AI Systems	ZZ Expectation Scan (QAOA-style Observable) — Oscillating Expectation Values	
⟨ZZ⟩ varies smoothly and periodically with the scan angle, revealing interference structure.
	Figure E.2.2


**E.2 Lab 6 — Simple VQE-Style Minimization (Toy) — Converging Energy Curve**



In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Beginner_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


In [ ]:
# === Environment Setup ===
import sys, subprocess, pkgutil
def ensure(pkg):
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p in ['qiskit','qiskit-aer','matplotlib','numpy','scikit-learn']:
    ensure(p)
import qiskit, numpy as np, matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print('Qiskit:', qiskit.__version__)


---

**Lab 11: Simple VQE-Style Minimization (Toy) — Converging Energy Curve**

**Searching for Ground States with Circuits**

This lab mimics how Variational Quantum Eigensolvers (VQE) find the lowest energy of a quantum system. By sweeping through parameter angles, the ZZ expectation value is calculated, and the plot identifies the “best angle” where the system achieves its minimum energy. The flow is: define a parameterized ansatz, run the circuit for different values, compute energies, and select the lowest. For undergraduates, the visualization shows how quantum algorithms don’t magically “know” the solution — they probe landscapes systematically, similar to gradient descent in classical optimization. The toy model here builds intuition for why VQE is central to simulating molecules and materials.

*Book Reference: Chapter 8 — Optimization and Control in Quantum AI Systems*

Readers explore the iterative search for minimum energy using parameterized circuits. This supports Chapter 8’s coverage of variational optimization loops and hybrid QAIS workflows.


**Expected Results**

⟨ZZ⟩ varies smoothly between −1 and 0 under this ansatz.

The minimum occurs near θ ≈ 0 or θ ≈ π, where the state approaches ∣01⟩ or ∣10⟩, yielding ⟨ZZ⟩ ≈ −1.

Mid-range θ values mix parities and raise the energy toward 0.

The dashed line at −1 marks the ground energy.

In [ ]:
# Lab 11 — VQE Toy Minimization: Minimize <ZZ> with a 1-parameter ansatz

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

# Observable (toy Hamiltonian): H = Z ⊗ Z
Z0Z1 = SparsePauliOp.from_list([("ZZ", 1.0)])

# 1-parameter product ansatz that sweeps parity:
# q0 gets RY(θ), q1 gets RY(π − θ) so that <ZZ> ∈ [−1, 0]
def ansatz(theta: float) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.ry(theta, 0)
    qc.ry(np.pi - theta, 1)
    return qc

# Sweep θ and compute exact expectation <ZZ>(θ)
angles = np.linspace(0.0, np.pi, 41)
vals = []
for th in angles:
    sv = Statevector.from_instruction(ansatz(th)).data
    exp = float((sv.conj() @ Z0Z1.to_matrix() @ sv).real)
    vals.append(exp)

# Find best (lowest energy) angle
best_idx = int(np.argmin(vals))
best_theta = float(angles[best_idx])
best_val = float(vals[best_idx])
print(f"Best angle (rad): {best_theta:.3f}   <ZZ> = {best_val:.3f}")

# Plot energy landscape
plt.plot(angles, vals, marker="o")
plt.axhline(-1.0, ls="--", lw=1, label="Ground energy = −1")
plt.xlabel("θ (radians)")
plt.ylabel("⟨ZZ⟩")
plt.title("VQE Toy Minimization: ⟨ZZ⟩ vs θ")
plt.legend()

# Save with correct IEEE label (single save)
fig, ax = plt.gcf(), plt.gca()
save_e_figure("Figure E.1.11", "P1_Lab11_VQE_Toy_Min.png",
              subdir="Beginner_Labs/figures", fig=fig, ax=ax)

plt.show()


*Figure E.1.11 — VQE Toy Minimization of ⟨ZZ⟩*
The curve shows the expectation value of Z ⊗ Z as a function of a single parameter θ in a product ansatz. The minimum reaches the ground energy −1 when the state concentrates on odd-parity components (∣01⟩ or ∣10⟩), demonstrating how a simple variational sweep can locate the optimum.

**Methodology Analysis**

We pose a toy VQE task with Hamiltonian H = Z ⊗ Z, whose ground energy is −1 at odd-parity eigenstates ∣01⟩ and ∣10⟩. We adopt a single-parameter ansatz: apply RY(θ) to qubit 0 and RY(π−θ) to qubit 1. For θ ∈ [0, π] we compute the exact expectation ⟨ZZ⟩ with the statevector simulator and select the angle that minimizes ⟨ZZ⟩. This emulates the inner loop of VQE without classical optimization code.

**Technical Analysis (for the Visual)**

For product states with RY rotations, the local Z expectations are ⟨Z₀⟩ = cos θ and ⟨Z₁⟩ = cos(π−θ) = −cos θ, and the ZZ expectation (since there are no correlations beyond product) becomes
⟨ZZ⟩ = ⟨Z₀⟩⟨Z₁⟩ = −cos² θ ∈ [−1, 0].
Thus, odd parity is favored at the minimum where cos² θ = 1. Although this is a simple, non-entangling ansatz, it cleanly demonstrates the variational principle: by tuning parameters, we minimize the expectation value of H and recover the ground state energy. In practice, richer Hamiltonians use expressive ansätze and classical optimizers, but the diagnostic shape of the energy landscape here mirrors real VQE behavior.

**Intuition Sidebar — “Steering into the valley”**

Imagine a hiker with one knob that steers left or right along a hillside. Turning the knob changes which side of the ridge you favor. At one extreme you stand squarely in the valley bottom (energy −1); away from it you climb back up toward the ridge (energy 0). VQE turns knobs like this to find valleys in an energy landscape.


---

**Conclusion — Lab 11**

This lab demonstrates the variational principle in its simplest form. By sweeping a single parameter, you minimized the expectation of H = Z ⊗ Z and recovered the ground energy −1 at odd-parity states. The exercise illustrates why good ansatz design matters: it must allow the optimizer to explore the subspace that contains the true minimum. Even a compact product ansatz can solve the toy task and build intuition for larger VQE problems.

**Key Takeaways**

The ground state of Z ⊗ Z has energy −1 at odd parity.

A thoughtfully chosen 1-parameter ansatz can reach the optimum; ansatz choice governs what energies are reachable.

The energy landscape ⟨H⟩(θ) provides immediate, visual feedback about expressivity and minima, mirroring real VQE workflows.

---
**Overall Summary (Labs 1–11)**

Across the eleven labs, readers progress from the foundations of quantum information (superposition, entanglement, and gate operations in Labs 1–2), through data encoding and reasoning architectures (Labs 3–6), into benchmarking and optimization methods (Labs 7–9), and finally into protocols and applied workflows (Labs 10–11). This trajectory mirrors the book’s structure, ensuring that practical experimentation reinforces conceptual mastery.

When contrasted with classical computing, the distinction becomes clear:

Classical methods (e.g., logistic regression) depend on fixed linear separations and explicit features.

Quantum methods embed data into high-dimensional Hilbert spaces, enabling similarity kernels, entangled correlations, and variational energy landscapes that classical baselines cannot access.

The visualization styles highlight these contrasts:

Bar charts capture quantum probabilities constrained by measurement.

Heatmaps reveal similarity structures induced by quantum encodings.

Smooth curves trace variational landscapes and optimization trajectories.

Together, the labs demonstrate how quantum computing extends and enriches classical ideas. In some cases, quantum approaches generalize familiar tasks such as classification and optimization; in others, they introduce fundamentally new tools, such as fidelity measures, teleportation protocols, and tamper-evident entanglement checks.

This integrated arc equips readers with both hands-on intuition and professional-level insight into how quantum methods reshape AI architectures and computational practice.

---

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 8 — Optimization and Control in Quantum AI Systems**:  
- Questions 4 and 5 (variational principle and energy minimization).  
They extend the variational analysis validated in **E.2 Lab 6**, where energy converges to the ground state minimum (Figure E.2.6).


**Congratulations — Lab 11**

Congratulations on completing Lab 11 — VQE Toy Minimization! You implemented a compact variational sweep, analyzed the energy landscape, and reached the ground state energy for a simple Hamiltonian. These are the core muscles behind VQE and quantum optimization. As you scale to richer Hamiltonians and more expressive ansätze, this disciplined approach to ansatz design and measurement-driven diagnostics will continue

**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---
